# Kuvageneraattorisovellusten rakentaminen (Azure OpenAI)

Suurissa kielimalleissa on enemmänkin tarjottavaa kuin tekstin generointi. Voit myös luoda kuvia tekstikuvausten perusteella. Kuvamuoto on hyödyllinen esimerkiksi lääketieteessä, arkkitehtuurissa, matkailussa, pelikehityksessä, markkinoinnissa ja monissa muissa. Tässä oppitunnissa työskentelemme tämän päivän **GPT Image** -mallien kanssa Microsoft Foundryssä.

## Oppimistavoitteet

Oppitunnin jälkeen osaat:

- Selittää, mitä kuvagenerointi on ja missä sitä voi hyödyntää.
- Ymmärtää `gpt-image` -malliperheen ja miten se eroaa legacy-mallisesta DALL·E -malleista.
- Rakentaa kuvageneraattorisovelluksen ja muokata kuvia.

## Mitä on kuvagenerointi?

Kuvagenerointimallit luovat kuvia tekstikehotteen pohjalta. Nykyaikaiset mallit, kuten `gpt-image`, oppivat koulutuksen aikana tekstin ja kuvien välisen yhteyden, ja muuntavat toistuvasti satunnaisen kohinan kuviksi, jotka vastaavat kehotettasi.

Kaksi tunnettua kuvamalliperhettä ovat:

- **`gpt-image` (OpenAI)** — tämän oppitunnin nykyinen sukupolvi. Se tukee tekstistä kuvaan -generointia ja kuvien muokkausta (inpainting maskin avulla).
- **Midjourney** — suosittu kolmannen osapuolen malli, jolla on oma palvelu ja Discord-pohjainen työnkulku.

> Vanhemmat OpenAI-kuvamallit — **DALL·E 2** ja **DALL·E 3** — ovat legacy-malleja. DALL·E 3 ei ole enää käytettävissä uusissa käyttöönotossa, ja ominaisuudet kuten `create_variation` olivat vain DALL·E 2:ssa. Käytä `gpt-image`-malleja uusissa sovelluksissa.

Microsoft Foundryssä **`gpt-image-2`** on uusin ja tehokkain kuvamalli, ja se on suositeltu oletus. `gpt-image-1.5` ja `gpt-image-1-mini` ovat myös yleisesti saatavilla.

> **Tärkeää:** `gpt-image` -mallit palauttavat luodun kuvan **base64** -muodossa (`b64_json`), eivät URL-osoitteena. Koodisi dekoodaa base64-merkkijonon tavuiksi ja tallentaa sen — kuvaa ei ole ladattavissa URL-osoitteena.


## Ensimmäisen kuvageneraattorisovelluksesi rakentaminen

Mitä tarvitaan kuvageneraattorisovelluksen rakentamiseen? Tarvitset seuraavat kirjastot:

- **python-dotenv**, suosittelemme vahvasti tämän kirjaston käyttöä pitämään salaisuutesi *.env*-tiedostossa erillään koodista.
- **openai**, tätä kirjastoa käytät kommunikoimaan OpenAI API:n kanssa.
- **pillow**, kuvien käsittelyyn Pythonissa.

Jos tätä ei ole vielä tehty, seuraa ohjeita [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst) -sivulla luodaksesi Microsoft Foundry -resurssin ja mallin. Valitse **gpt-image-2** malliksi (uusin Azure OpenAI -kuvamalli; DALL·E on vanhentunut).

1. Luo tiedosto *.env* seuraavalla sisällöllä:

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    Löydät nämä tiedot Microsoft Foundry -portaalista resurssiltasi "Käyttöönotot" ("Deployments") -osiosta.



1. Kerää yllä olevat kirjastot tiedostoon nimeltä *requirements.txt* seuraavasti:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Luo seuraavaksi virtuaaliympäristö ja asenna kirjastot:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Windowsilla käytä seuraavia komentoja virtuaaliympäristön luomiseen ja aktivoimiseen:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Lisää seuraava koodi tiedostoon nimeltä *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # importoi dotenv
    dotenv.load_dotenv()

    # Konfiguroi Azure OpenAI (Microsoft Foundry) -asiakas.
    # Kuvamallit tarvitsevat uuden API-version — tarkista Foundryn dokumentaatiosta, mikä versio mallillesi vaaditaan.
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # Luo kuva käyttämällä kuvageneraation API:a
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Syötä kehotetekstisi tähän
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # esim. "gpt-image-2"
        )
        # Aseta tallennetulle kuvalle hakemisto
        image_dir = os.path.join(os.curdir, 'images')

        # Jos hakemistoa ei ole olemassa, luo se
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Alusta kuvan polku (huomioi, että tiedostotyyppi on png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image-mallit palauttavat kuvan base64-muodossa (b64_json), eivät URL-osoitteena
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Näytä kuva oletuskuvankatseluohjelmassa
        image = Image.open(image_path)
        image.show()

    # käsittele poikkeukset
    except BadRequestError as err:
        print(err)

    ```

Selitetään tämä koodi:

- Ensin tuomme tarvitsemamme kirjastot, mukaan lukien OpenAI-kirjaston, dotenv-kirjaston, base64-moduulin ja Pillow-kirjaston.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- Seuraavaksi luemme ympäristömuuttujat *.env*-tiedostosta.

    ```python
    # tuo dotenv
    dotenv.load_dotenv()
    ```

- Tämän jälkeen määrittelemme Azure OpenAI:n (Microsoft Foundry) asiakasohjelman.

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- Seuraavaksi generoimme kuvan:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Syötä kehotetekstisi tähän
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    `gpt-image` mallien palauttama kuva on **base64**-merkkijono kohdassa `data[0].b64_json`. Dekoodamme sen tavuiksi ja kirjoitamme tiedostoon — ladattavaa URL-osoitetta ei ole.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Lopuksi avaamme kuvan ja käytämme tavallista kuvan katseluohjelmaa sen näyttämiseen:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Lisätietoja kuvan generoinnista

Tarkastellaan `images.generate`-funktion parametreja:

- **prompt**, on tekstipohja kuvan generointiin. Tässä se on "Pupu hevosen selässä, pitäen tikkaria, sumuisella niityllä missä kasvaa narsisseja".
- **size**, on generoiden kuvan koko (esimerkiksi `1024x1024`, `1536x1024`, `1024x1536` tai `"auto"`).
- **n**, on generoitavien kuvien määrä. Tässä generoimme yhden.
- **model**, on kuvamallisi käyttöönoton nimi (esimerkiksi `gpt-image-2`).

> Kuvamallit eivät käytä `temperature`-parametria — se koskee tekstin generointia. Vaihtelun saamiseksi kutsu API uudelleen; vaihtelun vähentämiseksi tee promptista tarkempi.

## Kuvageneroinnin lisäominaisuudet

Olet nähnyt, miten kuva generoidaan muutamalla Python-rivillä. `gpt-image`-mallit voivat myös **muokata** olemassa olevaa kuvaa. Antamalla kuvan, valinnaisen **maskin** (joka merkitsee alueen, jota muokataan) ja promptin, voit muuttaa osaa kuvasta — esimerkiksi lisätä pupullemme hatun.

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# muokkaukset palautetaan myös base64-muodossa
edited_image = base64.b64decode(response.data[0].b64_json)
```

Peruskuvassa on vain kani; lopullisessa kuvassa kanilla on hattu.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, otathan huomioon, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäinen asiakirja sen alkuperäiskielellä on virallinen lähde. Tärkeissä asioissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
